In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
import seaborn as sns
from rapidfuzz import fuzz, process
from sklearn.model_selection import cross_validate
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed

from scholarlm.utils import (
    load_and_process_results,
    match_datasets,
    matching_precision_recall,
    get_filenames_in_directory,
    fit_temperature,
    apply_temperature,
    fit_temperature_from_probs,
    apply_temperature_from_probs,
)
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

/projectnb/mcnet/kevin/coastal/scholarlm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 03-30 08:32:22 [__init__.py:216] Automatically detected platform cuda.


### Ground Truth Dataset

In [10]:
ground_truth_df = pd.read_csv("../data/nfix/meta/aquatic_N2fix_rates.csv")
ground_truth_df = ground_truth_df.loc[ground_truth_df.reference_id.isin(registered_ids)]

In [8]:
ground_truth_df.columns

Index(['nfix_rate_id', 'reference_id', 'site_name', 'latitude', 'longitude',
       'habitat', 'year', 'month', 'day', 'hour_minute', 'season', 'substrate',
       'substrate_details', 'sample_depth', 'nfix_rate_converted',
       'nfix_error_converted', 'nfix_unit_converted', 'nfix_error_type',
       'nfix_n', 'nfix_rate_original', 'nfix_error_original',
       'nfix_unit_original', 'nfix_method', 'ARA_conversion_factor',
       'nfix_incubation_time', 'nfix_incubation_temp'],
      dtype='object')

In [11]:
ground_truth_df.nfix_unit_converted.value_counts()

nfix_unit_converted
umol-n m-2 h-1    512
umol-n l-1 h-1    297
umol-n g-1 h-1    149
umol-n m-3 h-1     33
Name: count, dtype: int64

In [4]:
# ---------------------------------
# Load from ground truth dataset
# ---------------------------------

# Directory
with open(os.path.join("../data/nfix/directory.json"), "r") as f:
    paper_info = json.load(f)

def text_or_table_extraction(location):
    if 'figure' in location:
        return False
    if 'supplement' in location:
        return False
    if 'archive' in location:
        return False
    if 'author' in location:
        return False
    else:
        return True

registered_paper_info = {
    R: Rinfo for R,Rinfo in paper_info.items() if text_or_table_extraction(Rinfo['extraction_location']) 
}
registered_ids = list(registered_paper_info.keys())

#ground_truth_df = pd.read_csv("../data/nfix/nfix_data_corrected.csv", encoding_errors='ignore', index_col = 0)
#ground_truth_df = ground_truth_df.loc[ground_truth_df.title.isin(registered_titles)]
#ground_truth_df = ground_truth_df.reset_index(drop=True)

ground_truth_df = pd.read_csv("../data/nfix/meta/aquatic_N2fix_rates.csv")
ground_truth_df = ground_truth_df.loc[ground_truth_df.reference_id.isin(registered_ids)]

id_cols = [
        'nfix_rate_id', 'reference_id', 'site_name', 'latitude', 'longitude',
        'habitat', 'year', 'month', 'day', 'hour_minute', 'season',
        'substrate', 'substrate_details'
]

# Build a list of records, one per attribute
records = []

# 1) nfix_rate_converted -> attribute="rate_converted"
#    error from nfix_error_converted, error_type from nfix_error_type,
#    units from nfix_unit_converted
'''
records.append(nfix_df[id_cols].assign(
    attribute='rate_converted',
    value=nfix_df['nfix_rate_converted'],
    error=nfix_df['nfix_error_converted'],
    error_type=nfix_df['nfix_error_type'],
    units=nfix_df['nfix_unit_converted'],
))
'''

# 2) nfix_rate_original -> attribute="rate_original"
#    error from nfix_error_original, error_type from nfix_error_type,
#    units from nfix_unit_original
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_rate',
    value=ground_truth_df['nfix_rate_original'],
    error=ground_truth_df['nfix_error_original'],
    error_type=ground_truth_df['nfix_error_type'],
    units=ground_truth_df['nfix_unit_original'],
))

# 3) sample_depth -> attribute="sample_depth"
records.append(ground_truth_df[id_cols].assign(
    attribute='sample_depth',
    value=ground_truth_df['sample_depth'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

# 4) nfix_method -> attribute="method"
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_method',
    value=ground_truth_df['nfix_method'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

'''
# 5) nfix_incubation_time -> attribute="incubation_time"
records.append(nfix_df[id_cols].assign(
    attribute='incubation_time',
    value=nfix_df['nfix_incubation_time'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))
'''

# 6) nfix_incubation_temp -> attribute="incubation_temp"
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_incubation_temp_temperature',
    value=ground_truth_df['nfix_incubation_temp'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

ground_truth_df = pd.concat(records, ignore_index=True)

# Reorder columns
ground_truth_df = ground_truth_df[id_cols + ['attribute', 'value', 'error', 'error_type', 'units']]
ground_truth_df = ground_truth_df.dropna(subset=['value'])

# Optionally sort so each original row's attributes are grouped together
ground_truth_df = ground_truth_df.sort_values(id_cols, kind='mergesort').reset_index(drop=True)

ground_truth_df = ground_truth_df.loc[ground_truth_df.attribute == 'nfix_rate']
ground_truth_df.reset_index(drop=True, inplace=True)

In [51]:
ground_truth_df.reference_id.value_counts()[:10]

reference_id
R163    121
R164     84
R172     42
R248     32
R124     31
R51      31
R59      26
R114     26
R43      25
R103     23
Name: count, dtype: int64

### Full, Extracted Dataset

In [14]:
# ---------------------------------
# Load experiment results
# ---------------------------------

experiment_data_path = "../data/experiments/2026_04_01/nfix_final.json"

unit_conversion_table = {
    'nfix_rate': {},
    'nfix_incubation_time': {},
    'nfix_incubation_temperature': {"°C": 1,},
}

attribute_types = {
    'nfix_rate_mass': float,
    'nfix_rate_areal': float,
    'nfix_rate_volumetric': float,
    'nfix_incubation_time': float,
    'nfix_incubation_temperature': float,
}

# NOTE: some of these things you should get rid of in your extraction process!
drop_keys = ["feature_terms", "attribute_terms", "abbreviations", "table_logprob", "page_logprob", "judgement_raw_text"]
drop_attrs = ['nfix_incubation_time', 'nfix_incubation_temperature', 'sample_depth']

extracted_df = load_and_process_results(
    json_path=experiment_data_path,
    unit_conversion_table=unit_conversion_table,
    attribute_types=attribute_types,
    drop_keys=drop_keys,
    drop_attrs=drop_attrs,
    attribute_col="attribute",
    value_col="value",
    unit_col="units",
    out_col="processed_value"
)



# NOTE you need to change this to 'attribute'
#extracted_df.rename(columns={"feature": "attribute"}, inplace=True)
extracted_df.sort_values(by=["title", "attribute"], inplace=True)
extracted_df.reset_index(drop=True, inplace=True)
extracted_df.attribute = "nfix_rate"

#xtracted_df = extracted_df.loc[extracted_df.attribute == 'nfix_rate']
#extracted_df = extracted_df.reset_index(drop=True)

In [18]:
extracted_df.loc[extracted_df.paper_code == "R163"]

,reference,doi,doi_data,url,publication_year,authors,title,publication,volume,pages,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value
1081,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,837.90,μmol N m^{-2} d^{-1},[0],[None],[None],[None],[text],[Rates of dinitrogen fixation and the abundanc...,655,837.90
1082,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,2.00,nmol N L−1 d−1,[9],[None],[None],[None],[text],[Fig. 3. Areal rates of N_2 fixation (\( \mu \...,656,2.00
1083,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,40.90,μmol N m−2 d−1,[14],[None],[None],[None],[text],"[Table 8. Continued.\n\n<table number=""9"">\n<c...",657,40.90
1084,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,3.70,3.7,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",658,3.70
1085,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,62.50,nmol_n_l_d,[5],[3],"[('MAS 1', 'near_surface')]",[n2_fixation_nmol_n_l_d],[table],"[<table number=""3"">\n<caption>Physical and che...",659,62.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1740,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,51.70,nmol N L−1 d−1,[5],[3],"[('GSI 2', 'near_surface')]",[n2_fixation_nmol_n_l_d],[table],"[<table number=""3"">\n<caption>Physical and che...",3499,51.70
1741,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,0.20,nmol_l_d,[6],[4],"[('GSI 2', '23.9')]",[n2_fixation_nmol_l_d],[table],"[<table number=""4"">\n<caption>Physical and che...",3500,0.20
1742,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,8.60,nmol N L−1 d−1,[7],[5],"[('−74.642', 'MAS')]",[n2_fixation_nmol_n_l_d],[table],"[<table number=""5"">\n<caption>Physical and che...",3501,8.60
1743,Mulholland et al 2012 L and O 57,10.4319/lo.2012.57.4.1067,NaN,http://doi.wiley.com/10.4319/lo.2012.57.4.1067,2012,"Mulholland, M. R.; Bernhardt, P. W.; Blanco-Ga...",Rates of dinitrogen fixation and the abundance...,Limnology and Oceanography,57,1067-1083,...,2.70,nmol N L−1 d−1,[8],[6],"[('−74.642', 'MAS')]",[n2_fixation_nmol_n_l_1_d_1],[table],"[<table number=""6"">\n<caption>Physical and che...",3502,2.70


In [12]:
extracted_df.sample_depth

0       surface
1          None
2          None
3          None
4          None
         ...   
1881       None
1882       None
1883       None
1884       None
1885       None
Name: sample_depth, Length: 1886, dtype: object

In [15]:
extracted_df.attribute.value_counts()

attribute
nfix_rate    2079
Name: count, dtype: int64

### Match Extractions to Ground Truth

In [16]:
# Set of attributes which must be strictly equivalent to create a match
strict_matching = {
    "reference_id": "paper_code", # name in the ground truth dataset : name in the extracted dataset
    "attribute": "attribute",
    "value": "processed_value"
}

# Set of attributes which should be 
# compared by a fuzzy matching (roughly similar) to create a match.
fuzzy_matching = {
    "site_name": "name",
    #"substrate": "substrate_type",
    #"habitat": "habitat_type",
}

# This can take a while to run if you have a lot of data, 
# since it compares every extracted row to every ground truth row.
matching, matching_recall, matching_precision = matching_precision_recall(
    ground_truth_df,
    extracted_df,
    strict_matching=strict_matching,
    fuzzy_matching=fuzzy_matching,
    fuzzy_threshold = 0.0
)

print(f"Recall: {matching_recall:.4f}")
print(f"Precision: {matching_precision:.4f}")

Recall: 0.2260
Precision: 0.1077


### Debugging

In [20]:
gt_matched = np.array([False] * ground_truth_df.shape[0])
ex_matched = np.array([False] * extracted_df.shape[0])
for gt_idx, ex_idx in matching:
    gt_matched[gt_idx] = True
    ex_matched[ex_idx] = True

unmatched_gt = np.where(~gt_matched)[0]
unmatched_ex = np.where(~ex_matched)[0]

matched_gt_df = ground_truth_df[gt_matched == True]
unmatched_gt_df = ground_truth_df[gt_matched == False]
unmatched_gt_refs = unmatched_gt_df.reference_id.value_counts().index

matched_ex_df = extracted_df[ex_matched == True]
unmatched_ex_df = extracted_df[ex_matched == False]
unmatched_ex_refs = unmatched_ex_df.paper_code.value_counts().index

In [22]:
unmatched_gt_df.reference_id.value_counts()

reference_id
R164    84
R163    67
R172    38
R51     31
R124    26
        ..
R144     1
R221     1
R78      1
R264     1
R250     1
Name: count, Length: 89, dtype: int64

In [24]:
ref = "R163"
gt_ref_df = ground_truth_df.loc[ground_truth_df.reference_id == ref]
unmatched_gt_ref_df = unmatched_gt_df.loc[unmatched_gt_df.reference_id == ref]
ex_ref_df = extracted_df.loc[extracted_df.paper_code == ref]
unmatched_ex_ref_df = unmatched_ex_df.loc[unmatched_ex_df.paper_code == ref]

In [29]:
unmatched_gt_ref_df

,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
523,N3722,R163,east coast USA,37.519,-75.050,continental shelves,2006.0,7.0,1.0,NaN,NaN,wc,water column,nfix_rate,12.9,0.4,stdev.,nmol-n l-1 d-1
525,N3724,R163,east coast USA,36.627,-74.863,continental shelves,2006.0,7.0,1.0,NaN,NaN,wc,water column,nfix_rate,10.8,2.1,stdev.,nmol-n l-1 d-1
526,N3725,R163,east coast USA,36.601,-75.058,continental shelves,2006.0,7.0,1.0,NaN,NaN,wc,water column,nfix_rate,11.7,0.9,stdev.,nmol-n l-1 d-1
527,N3726,R163,east coast USA,36.532,-75.273,continental shelves,2006.0,7.0,1.0,NaN,NaN,wc,water column,nfix_rate,17.3,1.9,stdev.,nmol-n l-1 d-1
529,N3728,R163,east coast USA,36.900,-75.908,continental shelves,2006.0,7.0,1.0,NaN,NaN,wc,water column,nfix_rate,24.5,6.9,stdev.,nmol-n l-1 d-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
632,N3829,R163,east coast USA,40.607,-68.529,continental shelves,2009.0,8.0,1.0,NaN,NaN,wc,water column,nfix_rate,4.5,0.2,stdev.,nmol-n l-1 d-1
634,N3830,R163,east coast USA,41.404,-68.214,continental shelves,2009.0,8.0,1.0,NaN,NaN,wc,water column,nfix_rate,4.7,0.5,stdev.,nmol-n l-1 d-1
635,N3831,R163,east coast USA,41.268,-67.822,continental shelves,2009.0,8.0,1.0,NaN,NaN,wc,water column,nfix_rate,12.2,1.2,stdev.,nmol-n l-1 d-1
636,N3832,R163,east coast USA,40.984,-67.203,continental shelves,2009.0,8.0,1.0,NaN,NaN,wc,water column,nfix_rate,5.6,1.0,stdev.,nmol-n l-1 d-1


In [35]:
unmatched_ex_ref_df.loc[unmatched_ex_ref_df.table_number == 1]

,reference,doi,doi_data,url,publication_year,authors,title,publication,volume,pages,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value


In [43]:
table1_unmatched_df = unmatched_ex_ref_df[unmatched_ex_ref_df['table_number'].apply(lambda x: 1 in x)]

In [50]:
table1_unmatched_df.loc[table1_unmatched_df.value == 3.7].iloc[:, 14:]

,name,location,latitude,longitude,habitat_type,substrate_type,sample_depth,year,month,day,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value
1098,Mid-Atlantic Shelf,"37.519°N, −75.050°W",37.519,-75.050,shelf,water,29,2006.0,7.0,NaN,...,3.7,unitless,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",704,3.7
1105,Mid-Atlantic Shelf,"38.042°N, −74.299°W",38.042,-74.299,shelf,water,None,2006.0,7.0,NaN,...,3.7,umol N m−2 d−1,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",727,3.7
1135,Mid-Atlantic Shelf,"38.042°N, −74.299°W",38.042,-74.299,shelf,water,None,2006.0,7.0,NaN,...,3.7,3.7,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",886,3.7
1149,Mid-Atlantic Shelf,"37.695°N, −75.298°W",37.695,-75.298,shelf,water,None,2006.0,10.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",935,3.7
1156,Mid-Atlantic Shelf,"37.631°N, −75.152°W",37.631,-75.152,shelf,water,None,2006.0,10.0,NaN,...,3.7,3.7,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",957,3.7
1163,Mid-Atlantic Shelf,"37.519°N, −75.050°W",37.519,-75.050,shelf,water,None,2006.0,10.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",981,3.7
1199,Mid-Atlantic Shelf,"36.554°N, −75.706°W",36.554,-75.706,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1117,3.7
1206,Mid-Atlantic Shelf,"37.428°N, −75.476°W",37.428,-75.476,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1142,3.7
1230,Mid-Atlantic Shelf,"36.417°N, −75.700°W",36.417,-75.700,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1261,3.7
1237,Mid-Atlantic Shelf,"36.417°N, −75.700°W",36.417,-75.700,shelf,water,None,2009.0,8.0,NaN,...,3.7,umol N m−2 d−1,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1285,3.7


In [46]:
table1_unmatched_df.units.value_counts()

units
nmol N L−1 d−1    28
3.7               17
umol N m−2 d−1    13
nmol N            10
umol-N m-2 d-1     5
n2_fixation        1
unitless           1
3.8                1
nmol/L/d           1
nmol L-1 d-1       1
nmol L−1 d−1       1
Name: count, dtype: int64